# 🔋 Fuel Cell Engine — Ni@X Catalyst Screening

**Pipeline:** CHGNet filter → OC20 HOR activity → MACE NEB/MD → BoTorch optimize → JDFTx validate

**Runtime:** T4 GPU (16GB) — chọn Runtime → Change runtime type → T4 GPU

**Estimated time:** 15-30h full pipeline (chạy qua 2-3 sessions, auto-resume)

---

## 1. Setup Environment

In [ ]:
# Mount Google Drive (save checkpoint + results)
from google.colab import drive
drive.mount('/content/drive')

# Create project folder on Drive
!mkdir -p /content/drive/MyDrive/fuel-cell-engine/results
print('✅ Drive mounted')

In [ ]:
# Install dependencies — Colab T4 (torch 2.x + CUDA 12.x)
# IMPORTANT: After this cell → Runtime → Restart session → re-run from cell 1
import subprocess, torch

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda.replace('.','')[:3] if torch.version.cuda else 'cpu'
PYG_URL   = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'
print(f'torch={TORCH_VER}  cuda={CUDA_VER}')

def run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        lines = [l for l in r.stderr.split('\n') if 'ERROR' in l or 'error' in l.lower()]
        msg = lines[-1] if lines else r.stderr.strip()[-200:]
        print(f'  ❌ {msg}')
        if check:
            raise RuntimeError(f'Install failed: {cmd[:60]}')
    return r.returncode == 0

print('0/7  Pin numpy 1.26.4 (MACE/CHGNet require numpy<2)...')
run('pip install -q "numpy==1.26.4"')

print('1/7  Core scientific stack...')
run('pip install -q pandas matplotlib pydantic tqdm ase pymatgen')

print('2/7  PyTorch Geometric (required by MACE)...')
run('pip install -q torch_geometric')
run(f'pip install -q torch_scatter torch_sparse -f {PYG_URL}', check=False)

print('3/7  e3nn (pinned 0.4.4) + MACE...')
run('pip install -q "e3nn==0.4.4"')
run('pip install -q mace-torch')

print('4/7  CHGNet...')
run('pip install -q chgnet')

print('5/7  BoTorch / GPyTorch...')
run('pip install -q botorch gpytorch')

print('6/7  fairchem deps...')
run('pip install -q torchtnt hydra-core omegaconf ray submitit lmdb orjson')
run('pip install -q fairchem-core --no-deps', check=False)

print('7/7  Anthropic...')
run('pip install -q anthropic')

print()
print('✅ Install done')
print('⚠️  MUST DO: Runtime → Restart session → re-run from cell 1 (skip this cell)')

In [ ]:
# Verify GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Verify all models load on GPU — run AFTER restart session
import torch

errors = []
print('=== VERIFY GPU MODELS ===')
print(f'torch={torch.__version__}  numpy={__import__("numpy").__version__}')
print(f'CUDA={torch.cuda.is_available()}  GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')

print()
print('Loading MACE...')
try:
    from mace.calculators import mace_mp
    calc = mace_mp(model='medium', device='cuda')
    print('✅ MACE loaded on GPU')
    del calc
except Exception as e:
    errors.append(f'MACE: {e}')
    print(f'❌ MACE failed: {e}')

print('Loading CHGNet...')
try:
    from chgnet.model import CHGNet
    model = CHGNet.load()
    print('✅ CHGNet loaded')
    del model
except Exception as e:
    errors.append(f'CHGNet: {e}')
    print(f'❌ CHGNet failed: {e}')

print('Loading fairchem/OC20...')
try:
    from fairchem.core import pretrained_mlip
    print('✅ fairchem loaded')
except Exception as e:
    print(f'⚠️  fairchem: {e}')
    print('   → OC20 will run in mock mode (acceptable for screening)')

torch.cuda.empty_cache()
print()

if errors:
    raise RuntimeError(
        f'❌ {len(errors)} required model(s) failed:\n' +
        '\n'.join(f'  - {e}' for e in errors) +
        '\n\nFix: re-run install cell → Restart session → re-run from cell 1'
    )
print('✅ All required models loaded — safe to proceed')

## 2. Upload Engine Code

**Option A:** Upload `fuel-cell-engine.zip` via Colab file browser (left panel)

**Option B:** Copy from Drive (nếu đã upload folder lên Drive trước)

In [ ]:
# Option A: Upload zip file
# 1. Zip folder fuel-cell-engine/ trên máy local
# 2. Upload file zip qua Colab sidebar (Files → Upload)
# 3. Uncomment và chạy:

# !unzip -q /content/fuel-cell-engine.zip -d /content/
# print('✅ Engine extracted')

# Option B: Copy from Drive
!cp -r /content/drive/MyDrive/fuel-cell-engine /content/engine 2>/dev/null || echo 'Upload engine code first!'

# Verify
!ls /content/engine/run.py 2>/dev/null && echo '✅ Engine code ready' || echo '❌ Engine not found — upload first'

In [ ]:
# Download databases (SynTERRA + GNoME)
import os
os.chdir('/content/engine')

# SynTERRA
if not os.path.exists('data/synterra_repo/solid-state_dataset_20200713.json'):
    print('Downloading SynTERRA...')
    !mkdir -p data
    !git clone --depth 1 https://github.com/CederGroupHub/text-mined-synthesis_public.git data/synterra_repo
    !cd data/synterra_repo && xz -dk solid-state_dataset_20200713.json.xz 2>/dev/null
    print('✅ SynTERRA ready')
else:
    print('✅ SynTERRA already exists')

# GNoME
if not os.path.exists('data/gnome_data/stable_materials_summary.csv'):
    print('Downloading GNoME (145MB)...')
    !mkdir -p data/gnome_data
    !curl -sL 'https://storage.googleapis.com/gdm_materials_discovery/gnome_data/stable_materials_summary.csv' \
        -o data/gnome_data/stable_materials_summary.csv
    !ls -lh data/gnome_data/stable_materials_summary.csv
    print('✅ GNoME ready')
else:
    print('✅ GNoME already exists')

## 3. Configuration

In [ ]:
import os
os.chdir('/content/engine')

# Claude API key (optional — skip nếu không có, engine vẫn chạy)
os.environ['ANTHROPIC_API_KEY'] = ''  # Điền hoặc để trống

# Chọn hướng chạy:
DIRECTION = 'A'  # 'A' = shell optimization, 'B' = bare alloy screening

if DIRECTION == 'A':
    CONFIG_PATH = 'rnd/experiments/carbon-shell-config.json'
    print('🔬 Hướng A: Ni@C Carbon Shell Optimization')
elif DIRECTION == 'B':
    CONFIG_PATH = 'rnd/experiments/alloy-screening-config.json'
    print('🔬 Hướng B: Ni-X Bare Alloy Screening')

print(f'Config: {CONFIG_PATH}')

## 4. Quick Test (5 min — verify everything works)

In [ ]:
# Quick test: chỉ 2 iterations (verify pipeline hoạt động trước khi chạy full)
# Sửa tạm n_iterations trong config
import json

with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

# Save original
original_iterations = config['optimizer']['n_iterations']
original_init = config['optimizer']['n_init']

# Quick test settings
config['optimizer']['n_iterations'] = 2
config['optimizer']['n_init'] = 3

TEST_CONFIG = '/content/engine/test_config.json'
with open(TEST_CONFIG, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Quick test config: {original_init}→3 init, {original_iterations}→2 iterations')
print('Running quick test (~5 min)...')

In [ ]:
!python run.py --config test_config.json --skip-calibration 2>&1 | tail -30

In [ ]:
# Verify quick test output
import os
output_dir = config['output']['dir']
print(f'Output dir: {output_dir}')
!ls -la {output_dir} 2>/dev/null || echo '❌ No output — check errors above'
print()
print('=== Ranked Candidates (Quick Test) ===')
!cat {output_dir}/ranked_candidates.md 2>/dev/null | head -30

## 5. Full Pipeline Run

⚠️ **Takes 15-30 hours on T4.** Colab may disconnect after 12h.

Checkpoint saves every iteration → resume nếu disconnect.

In [ ]:
# Symlink output to Drive (auto-save results)
import os

output_dir = config['output']['dir']
drive_output = '/content/drive/MyDrive/fuel-cell-engine/results/'
os.makedirs(drive_output, exist_ok=True)

# Copy checkpoint to Drive after each run
print(f'Results will be saved to: {drive_output}')
print(f'Config: {CONFIG_PATH}')
print(f'Iterations: {original_init} init + {original_iterations} BO')
print()
print('Starting full pipeline...')

In [ ]:
# FULL RUN (15-30h on T4)
!python run.py --config {CONFIG_PATH} \
    2>&1 | tee /content/drive/MyDrive/fuel-cell-engine/results/run.log

In [ ]:
# After completion: copy results to Drive
!cp -r {output_dir}/* /content/drive/MyDrive/fuel-cell-engine/results/
print('✅ Results saved to Drive')
!ls /content/drive/MyDrive/fuel-cell-engine/results/

## 6. Resume (Nếu Disconnect)

In [ ]:
# Resume after disconnect — re-install + restore environment
from google.colab import drive
drive.mount('/content/drive')

import subprocess, torch

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda.replace('.','')[:3] if torch.version.cuda else 'cpu'
PYG_URL   = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'

def run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        lines = [l for l in r.stderr.split('\n') if 'ERROR' in l or 'error' in l.lower()]
        msg = lines[-1] if lines else r.stderr.strip()[-200:]
        print(f'  ❌ {msg}')
        if check:
            raise RuntimeError(f'Install failed: {cmd[:60]}')
    return r.returncode == 0

print('Re-installing dependencies...')
run('pip install -q "numpy<2" pandas matplotlib pydantic tqdm ase pymatgen')
run('pip install -q torch_geometric')
run(f'pip install -q torch_scatter torch_sparse -f {PYG_URL}', check=False)
run('pip install -q "e3nn==0.4.4"')
run('pip install -q mace-torch')
run('pip install -q chgnet botorch gpytorch')
run('pip install -q fairchem-core --no-deps', check=False)
run('pip install -q submitit lmdb orjson anthropic')

# Verify MACE + CHGNet load
errors = []
try:
    from mace.calculators import mace_mp
    calc = mace_mp(model='medium', device='cuda'); del calc
    print('✅ MACE OK')
except Exception as e:
    errors.append(f'MACE: {e}')

try:
    from chgnet.model import CHGNet
    CHGNet.load()
    print('✅ CHGNet OK')
except Exception as e:
    errors.append(f'CHGNet: {e}')

if errors:
    raise RuntimeError('Models failed: ' + str(errors))

# Restore engine from Drive
import os
!cp -r /content/drive/MyDrive/fuel-cell-engine /content/engine
%cd /content/engine
os.environ['ANTHROPIC_API_KEY'] = ''  # re-fill if needed

print('✅ Environment restored — run Resume cell below')

In [ ]:
# Resume from checkpoint
DIRECTION = 'A'  # same as before

if DIRECTION == 'A':
    CONFIG_PATH = 'rnd/experiments/carbon-shell-config.json'
    CHECKPOINT = 'rnd/compute/ni_carbon_shell_optimization/checkpoint.pt'
elif DIRECTION == 'B':
    CONFIG_PATH = 'rnd/experiments/alloy-screening-config.json'
    CHECKPOINT = 'rnd/compute/ni_alloy_oxidation_screening/checkpoint.pt'

import os
if os.path.exists(CHECKPOINT):
    print(f'✅ Checkpoint found: {CHECKPOINT}')
    !python run.py --config {CONFIG_PATH} --resume {CHECKPOINT} \
        2>&1 | tee -a /content/drive/MyDrive/fuel-cell-engine/results/run.log
else:
    print(f'❌ No checkpoint at {CHECKPOINT}')
    print('Starting fresh...')
    !python run.py --config {CONFIG_PATH} \
        2>&1 | tee /content/drive/MyDrive/fuel-cell-engine/results/run.log

## 7. View Results

In [ ]:
# View ranked candidates
import os
if DIRECTION == 'A':
    OUTPUT_DIR = 'rnd/compute/ni_carbon_shell_optimization'
else:
    OUTPUT_DIR = 'rnd/compute/ni_alloy_oxidation_screening'

print('=== RANKED CANDIDATES ===')
!cat {OUTPUT_DIR}/ranked_candidates.md

In [ ]:
# View results table
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/results.csv')
print(f'Total candidates: {len(df)}')
print(f'Successful: {len(df[df["status"]=="success"])}')
print()
df.sort_values('hor_activity', key=abs).head(10)

In [ ]:
# View Pareto plot
from IPython.display import Image
Image(filename=f'{OUTPUT_DIR}/pareto.png')

In [ ]:
# View verification report
print('=== VERIFICATION REPORT ===')
!cat {OUTPUT_DIR}/verification_report.md

In [ ]:
# Final: copy all results to Drive
!cp -r {OUTPUT_DIR}/* /content/drive/MyDrive/fuel-cell-engine/results/
print('✅ All results saved to Google Drive')
print('📂 Path: /content/drive/MyDrive/fuel-cell-engine/results/')

---

## ⚠️ IMPORTANT NOTES

1. **Results = RANKING, not absolute numbers** (±0.15 eV uncertainty)
2. **Durability CANNOT be predicted** — must test in lab
3. **Trust Score ≥ B** → safe to proceed to lab validation
4. **Trust Score D/F** → do NOT proceed, debug first
5. **T4 slower than A100** (~5x) — be patient or split across sessions